In [ ]:
# Habitat Ranking Script

# Import Required Libraries
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rasterio_mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.geometry import Point
import numpy as np
from pathlib import Path


In [ ]:
# Set directories (update these paths to your actual file locations)
# You need a GeoTIFF landcover raster with pixel values representing landcover types,
# And a .csv file of GPS points with latitude and longitude columns.

RASTER_PATH = r"path\to\landcover.tif" # Update this to your landcover raster path
GPS_CSV_PATH = r"path\to\gps_points.csv" # Update this to your GPS points .csv path
OUTPUT = r"path\to\output" # Update this to your desired output directory

In [ ]:
# Load data and reproject GPS to match raster CRS

# Load GPS coordinates 
gps_df = pd.read_csv(GPS_CSV_PATH)
geometry = [Point(xy) for xy in zip(gps_df['decimalLongitude'], gps_df['decimalLatitude'])] # Update column names if different
gps_gdf = gpd.GeoDataFrame(gps_df, geometry=geometry, crs='EPSG:4326')

# Load raster and get its CRS
with rasterio.open(RASTER_PATH) as src:
    raster_crs = src.crs
    print(f"Raster CRS: {raster_crs}")

# Reproject GPS points to match raster CRS
gps_reprojected = gps_gdf.to_crs(raster_crs)

print(f"GPS points loaded: {len(gps_reprojected)}")
print(f"GPS reprojected to raster CRS: {raster_crs}")
print(f"Raster path: {RASTER_PATH}")


In [ ]:
# Buffering and analysis

# Create 75m buffers around each point
gps_reprojected['buffer'] = gps_reprojected.buffer(75) # Can change buffer size here if needed
buffers_gdf = gpd.GeoDataFrame(gps_reprojected[['buffer']], geometry='buffer', crs=gps_reprojected.crs)

# Extract landcover values within each buffer
landcover_counts = []

with rasterio.open(RASTER_PATH) as src:
    for idx, buffer_geom in enumerate(buffers_gdf.geometry):
        # Extract only the portion of raster within buffer bounds
        try:
            masked_data, masked_transform = rasterio_mask(src, [buffer_geom], crop=True, filled=False)
            buffer_values = masked_data[0]  # Get first band
            
            # Remove nodata/masked values
            if hasattr(buffer_values, 'mask'):
                buffer_values = buffer_values.compressed()  # Extract non-masked values
            if src.nodata is not None:
                buffer_values = buffer_values[buffer_values != src.nodata]
            
            # Count unique landcover types
            if len(buffer_values) > 0:
                unique, counts = np.unique(buffer_values, return_counts=True)
                
                for landcover_type, count in zip(unique, counts):
                    landcover_counts.append({
                        'buffer_id': idx,
                        'landcover_type': int(landcover_type),
                        'pixel_count': int(count)
                    })
        except ValueError:
            # Buffer doesn't intersect raster - skip it
            continue

landcover_counts_df = pd.DataFrame(landcover_counts)
print(f"Processed {len(buffers_gdf)} buffers")
print(f"Total landcover records: {len(landcover_counts_df)}")


In [ ]:
# Calculate statistics and write to CSV files
# This cell will take a while to calculate the total area for very large rasters (i.e. statewide)
# The raster is read in chunks to avoid memory issues

# Calculate overall proportions for each landcover type across all buffers
total_pixels_by_type = landcover_counts_df.groupby('landcover_type')['pixel_count'].sum().reset_index()
total_pixels_by_type.columns = ['landcover_type', 'buffer_pixels']
total_pixels_overall = total_pixels_by_type['buffer_pixels'].sum()
total_pixels_by_type['proportion'] = total_pixels_by_type['buffer_pixels'] / total_pixels_overall

# Save landcover counts with landcover_type and proportion
counts_output = f"{OUTPUT}\\landcover_counts.csv"
total_pixels_by_type[['landcover_type', 'proportion']].to_csv(counts_output, index=False)
print(f"Saved landcover counts to: {counts_output}")

# Calculate proportions for each buffer
landcover_counts_df['buffer_pixels'] = landcover_counts_df.groupby('buffer_id')['pixel_count'].transform('sum')
landcover_counts_df['proportion'] = landcover_counts_df['pixel_count'] / landcover_counts_df['buffer_pixels']

# Add latitude and longitude to landcover_counts_df by matching buffer_id to GPS points
gps_coords = gps_df[['decimalLatitude', 'decimalLongitude']].copy()
gps_coords['buffer_id'] = range(len(gps_coords))
landcover_counts_df = landcover_counts_df.merge(gps_coords, on='buffer_id', how='left')

proportions_output = f"{OUTPUT}\\landcover_proportions.csv"
landcover_counts_df[['buffer_id', 'decimalLatitude', 'decimalLongitude', 'landcover_type', 'proportion']].to_csv(proportions_output, index=False)
print(f"Saved landcover proportions to: {proportions_output}")

# Calculate RankOcc (landcover types by overall proportion across all buffers)
total_pixels_by_type['RankOcc'] = total_pixels_by_type['proportion'].rank(ascending=False, method='min').astype(int)
total_pixels_by_type = total_pixels_by_type.sort_values('RankOcc')

# Find dominant landcover type for each buffer
dominant_landcover = landcover_counts_df.loc[landcover_counts_df.groupby('buffer_id')['proportion'].idxmax()]
dominant_counts = dominant_landcover['landcover_type'].value_counts().reset_index()
dominant_counts.columns = ['landcover_type', 'buffers_as_dominant']

# Calculate total area for each landcover type (using pixel counts as proxy for area)
# Read raster in chunks to avoid memory issues
print("Calculating total area...")
type_counts_dict = {}

with rasterio.open(RASTER_PATH) as src:
    nodata = src.nodata
    # Read raster in windows (chunks)
    for ji, window in src.block_windows(1):
        raster_chunk = src.read(1, window=window)
        
        # Remove nodata values and 0 (NA) values
        if nodata is not None:
            raster_chunk = raster_chunk[raster_chunk != nodata]
        raster_chunk = raster_chunk[raster_chunk != 0]
        
        # Count unique values in this chunk
        unique_types, counts = np.unique(raster_chunk, return_counts=True)
        
        # Accumulate counts
        for landcover_type, count in zip(unique_types, counts):
            landcover_type = int(landcover_type)
            if landcover_type in type_counts_dict:
                type_counts_dict[landcover_type] += int(count)
            else:
                type_counts_dict[landcover_type] = int(count)

# Convert dictionary to DataFrame
total_area_df = pd.DataFrame(list(type_counts_dict.items()), 
                               columns=['landcover_type', 'raster_pixels'])

# Calculate overall_proportion as proportion in entire raster (excluding 0 and nodata)
total_raster_pixels = total_area_df['raster_pixels'].sum()
total_area_df['overall_proportion'] = total_area_df['raster_pixels'] / total_raster_pixels

# Calculate buffer_count: number of buffers each landcover type appears in
buffer_count = landcover_counts_df.groupby('landcover_type')['buffer_id'].nunique().reset_index()
buffer_count.columns = ['landcover_type', 'buffer_count']

# Merge 
rank_df = total_pixels_by_type.merge(dominant_counts, on='landcover_type', how='left')
rank_df['buffers_as_dominant'] = rank_df['buffers_as_dominant'].fillna(0)
rank_df = rank_df.merge(total_area_df, on='landcover_type', how='left')
rank_df = rank_df.merge(buffer_count, on='landcover_type', how='left')

# Calculate RankIndex (pixels within buffer area / total raster pixels)
rank_df['RankIndex'] = rank_df['buffer_pixels'] / rank_df['raster_pixels']

# Save final ranking file
ranking_output = f"{OUTPUT}\\landcover_rankings.csv"
rank_df.to_csv(ranking_output, index=False)
print(f"Saved landcover rankings to: {ranking_output}")

In [ ]:
# Extract landcover values at exact point locations for additional analysis

print("\nExtracting landcover values at exact point locations...")
point_landcover_values = []

with rasterio.open(RASTER_PATH) as src:
    for idx, point in enumerate(gps_reprojected.geometry):
        # Get pixel value at exact point location
        try:
            row, col = src.index(point.x, point.y)
            landcover_value = src.read(1, window=((row, row+1), (col, col+1)))[0, 0]
            
            # Handle nodata values
            if src.nodata is not None and landcover_value == src.nodata:
                landcover_value = None
            elif landcover_value == 0:
                landcover_value = 0
            else:
                landcover_value = int(landcover_value)
                
            point_landcover_values.append(landcover_value)
        except:
            # Point is outside raster bounds
            point_landcover_values.append(None)

# Create output dataframe with original GPS data, buffer_id, and point landcover
output_gps_df = gps_df.copy()
output_gps_df['buffer_id'] = range(len(output_gps_df))
output_gps_df['point_landcover_type'] = point_landcover_values

# Save GPS points with landcover data
points_output = f"{OUTPUT}\\Loammi_Points_With_Landcover.csv"
output_gps_df.to_csv(points_output, index=False)
print(f"Saved GPS points with landcover to: {points_output}")

print("\nAnalysis complete! Output files:")
print(f"  1. {counts_output}")
print(f"  2. {proportions_output}")
print(f"  3. {ranking_output}")
print(f"  4. {points_output}")